# Course 8: Research Engineering Interviews
## Session 5: ML from scratch II

**Goal:** write multi-head causal attention, a block, a tiny GPT, a training loop and sampling, and prove each correct.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### Tests
Write `MultiHeadAttention(d, n_heads)`, `Block(d, n_heads)`, `TinyGPT(vocab, d, n_heads, n_layers, block_size)` whose forward returns logits of shape (B, T, vocab), and `sample_top_p(logits, p, temperature)`.

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

def run_tests(MultiHeadAttention, TinyGPT, sample_top_p):
    torch.manual_seed(0)
    attn = MultiHeadAttention(32, 4)
    x = torch.randn(2, 7, 32); x2 = x.clone(); x2[:, -1] += 1.0
    assert attn(x).shape == (2, 7, 32)
    assert torch.allclose(attn(x)[:, :-1], attn(x2)[:, :-1], atol=1e-6), "attention is not causal"
    model = TinyGPT(vocab=50, d=32, n_heads=4, n_layers=2, block_size=16)
    ids = torch.randint(0, 50, (4, 16))
    logits = model(ids)
    assert logits.shape == (4, 16, 50)
    loss0 = F.cross_entropy(logits.view(-1, 50), ids.view(-1)).item()
    assert abs(loss0 - math.log(50)) < 0.5, loss0
    ids2 = ids.clone(); ids2[:, -1] = (ids2[:, -1] + 1) % 50
    assert torch.allclose(model(ids)[:, :-1], model(ids2)[:, :-1], atol=1e-5), "model is not causal"
    xb, yb = ids[:, :-1], ids[:, 1:]
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for _ in range(300):
        loss = F.cross_entropy(model(xb).reshape(-1, 50), yb.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    assert loss.item() < 0.1, loss.item()
    lg = torch.log(torch.tensor([0.5, 0.3, 0.15, 0.05]))
    assert {sample_top_p(lg, 0.9) for _ in range(500)} == {0, 1, 2}
    assert {sample_top_p(lg, 0.7) for _ in range(500)} == {0, 1}
    print(f"all tests pass | initial loss {loss0:.2f} vs ln 50 = {math.log(50):.2f} | overfit loss {loss.item():.4f}")

In [ ]:
class MultiHeadAttention(nn.Module):       # TODO (20 min, with a shape comment on every line)
    pass

# run_tests(MultiHeadAttention, TinyGPT, sample_top_p)

### Questions to answer aloud
- Why divide by the square root of the head width?
- Why is the mask applied before the softmax?
- Why pre-norm? What changes for an encoder?

---
## Reference solutions
Try every problem under the timer first. Then run this section: it defines working versions and runs the same tests.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                                   # each (B, T, D)
        split = lambda t: t.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)                                   # (B, H, T, D/H)
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // self.n_heads)          # (B, H, T, T)
        mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float("-inf"))
        out = torch.softmax(scores, dim=-1) @ v                                  # (B, H, T, D/H)
        return self.proj(out.transpose(1, 2).reshape(B, T, D))                   # (B, T, D)

class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d), MultiHeadAttention(d, n_heads)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, d, n_heads, n_layers, block_size):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(block_size, d)
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, vocab)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.size(1), device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln(x))                                             # (B, T, vocab)

def sample_top_p(logits, p=0.9, temperature=1.0):
    probs = torch.softmax(logits / temperature, dim=-1)
    sp, si = torch.sort(probs, descending=True)
    keep = torch.cumsum(sp, dim=-1) - sp < p
    sp = sp * keep
    return si[torch.multinomial(sp / sp.sum(), 1)].item()

run_tests(MultiHeadAttention, TinyGPT, sample_top_p)